# NF Augmentation Sweep Quickcheck

This notebook compares the focused augmentation sweep, with an optional Nick-default baseline loaded from `nf_sweep_v2` for reference:

| variant | augmentation pipeline | exact image operation |
|---|---|---|
| `nick_default` | previous `nf_sweep_v2` Nick-default baseline | reference model, not part of the augmentation sweep |
| `roll_flip` | `RandomRoll` + `RandomFlip` | periodic translation, then independent x/y reflections |
| `roll_flip_rot90` | `RandomRoll` + `RandomFlip` + `RandomRot90` | periodic translation, reflections, then a random 0/90/180/270 degree rotation |
| `roll_dihedral` | `RandomRoll` + `RandomDihedral2D` | periodic translation, then one random element of the D4 square-symmetry group |

Definitions for an image `x` with image dimensions `dims=(-2, -1)`:

```text
RandomRoll:       x[i, j] -> x[(i - dy) mod H, (j - dx) mod W]
RandomFlip:       with p=0.5 independently flip vertical and/or horizontal axes
RandomRot90:      x -> rot90(x, k), where k is uniformly one of {0,1,2,3}
RandomDihedral2D: x -> rot90(x, k), then optionally flip along the first image dim
```

`RandomDihedral2D` samples one of the 8 exact square symmetries: identity, rotations by 90/180/270 degrees, and the four reflected versions. These are exact index permutations: no interpolation, no smoothing, no amplitude changes.


## How To Run The Sweep

After training finishes, sample the runs before using the physics panels:

```bash
cd /home/jiamingp/diffusion_models_repo
python scripts/prepare_nf_sweep_aug_configs.py --project-dir "$PWD"
python scripts/prepare_nf_sweep_aug_configs.py --project-dir "$PWD" --check-only

# training, if not already submitted
sbatch -A huterer0 --array=0-2 scripts/slurm/train_nf_sweep_aug_array.sbatch

# samples for raw/EMA and train_full/DPM/DDIM
sbatch -A huterer0 scripts/slurm/sample_nf_sweep_aug_array.sbatch
```

Fast sample-only smoke test for raw `train_full` across the three variants:

```bash
sbatch -A huterer0 --array=0,18,36 scripts/slurm/sample_nf_sweep_aug_array.sbatch
```


In [ ]:
from __future__ import annotations

import json
import math
import os
import re
import sys
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_CANDIDATES = [
    Path.cwd(),
    Path('/home/jiamingp/diffusion_models_repo'),
    Path('/Users/apple/AI/Diffusion_model'),
]
PROJECT_DIR = next((p for p in PROJECT_CANDIDATES if (p / 'simdiff_eval').exists()), PROJECT_CANDIDATES[0])
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from simdiff_eval.io import as_nchw, load_real_from_config
from simdiff_eval.metrics import batch_power_spectra, field_histogram, power_spectrum_summary

MANIFEST_PATH = PROJECT_DIR / 'local/nf_sweep_aug/manifest.json'
CONFIG_DIR = PROJECT_DIR / 'local/nf_sweep_aug/configs'
CHECKPOINT_ROOT = Path(os.environ.get('NF_SWEEP_AUG_CHECKPOINT_ROOT', '/scratch/huterer_root/huterer0/jiamingp/saved_runs/nf_sweep_aug'))
SAMPLE_ROOT = PROJECT_DIR / 'results/nf_sweep_aug/samples'
OUTPUT_DIR = PROJECT_DIR / 'results/nf_sweep_aug/quickcheck'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = int(os.environ.get('NF_SWEEP_AUG_SEED', 123))
PREFERRED_SAMPLER = os.environ.get('NF_SWEEP_AUG_SAMPLER', 'train_full')
PREFERRED_EMA_LABEL = os.environ.get('NF_SWEEP_AUG_EMA_LABEL', 'raw')
MAX_RAW_REAL_CUBES = int(os.environ.get('NF_SWEEP_AUG_MAX_RAW_REAL_CUBES', 16))
MAX_REAL_HIST = int(os.environ.get('NF_SWEEP_AUG_MAX_REAL_HIST', 512))
MAX_REAL_PK = int(os.environ.get('NF_SWEEP_AUG_MAX_REAL_PK', 256))
MAX_GENERATED = int(os.environ.get('NF_SWEEP_AUG_MAX_GENERATED', 64))
PK_NBINS = int(os.environ.get('NF_SWEEP_AUG_PK_NBINS', 30))

VARIANT_ORDER = ['nick_default', 'roll_flip', 'roll_flip_rot90', 'roll_dihedral']
VARIANT_LABELS = {
    'nick_default': 'Nick default',
    'roll_flip': 'roll + flip',
    'roll_flip_rot90': 'roll + flip + rot90',
    'roll_dihedral': 'roll + dihedral',
}
EMA_LABELS = ['raw', 'ema0p16', 'ema0p18', 'ema0p20', 'ema0p22', 'ema0p25']
SAMPLER_LABELS = ['train_full', 'dpm25', 'ddim50']

INCLUDE_NICK_DEFAULT_BASELINE = os.environ.get('NF_SWEEP_AUG_INCLUDE_NICK_DEFAULT', '1') != '0'
NICK_DEFAULT_BASELINE = {
    'run_name': 'nf_sweep_v2_u128_n500_e100_nick_default',
    'arch': 'u128',
    'arch_label': 'U128',
    'variant_tag': 'nick_default',
    'variant_label': 'Nick default',
    'config': 'local/nf_sweep_v2/configs/nf_sweep_v2_u128_n500_e100_nick_default.yaml',
    'checkpoint_dir': '/scratch/huterer_root/huterer0/jiamingp/saved_runs/nf_sweep_v2/nf_sweep_v2_u128_n500_e100_nick_default_checkpoints',
    'sample_root': str(PROJECT_DIR / 'results/nf_sweep_v2/samples'),
    'epochs': 100,
    'note': 'Previous nf_sweep_v2 Nick-default u128 baseline; included for comparison, not an augmentation-sweep training job.',
}

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 180,
    'font.size': 12,
    'axes.titlesize': 13,
    'axes.labelsize': 12,
    'legend.fontsize': 9,
})

print('project:', PROJECT_DIR)
print('manifest:', MANIFEST_PATH, 'exists=', MANIFEST_PATH.exists())
print('checkpoint root:', CHECKPOINT_ROOT)
print('sample root:', SAMPLE_ROOT)
print('preferred sample:', PREFERRED_EMA_LABEL, PREFERRED_SAMPLER)


## Visual Toy Example: What Rot90 And Dihedral Do

This is a numbered toy image. The augmentation only reindexes pixels. Real fields get the same exact index permutation.


In [ ]:
toy = np.arange(16).reshape(4, 4)
examples = {
    'original': toy,
    'rot90 k=1': np.rot90(toy, 1),
    'rot90 k=2': np.rot90(toy, 2),
    'flip vertical': np.flip(toy, axis=0),
    'flip horizontal': np.flip(toy, axis=1),
    'dihedral example\nrot90+flip': np.flip(np.rot90(toy, 1), axis=0),
}
fig, axes = plt.subplots(1, len(examples), figsize=(2.2 * len(examples), 2.4))
for ax, (title, arr) in zip(axes, examples.items()):
    ax.imshow(arr, cmap='viridis')
    for i in range(arr.shape[0]):
        for j in range(arr.shape[1]):
            ax.text(j, i, str(arr[i, j]), ha='center', va='center', color='white', fontsize=9, weight='bold')
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])
fig.tight_layout()
plt.show()


## Discover Runs, Checkpoints, And Samples

If sample rows are missing, run the sampling SLURM script after training completes. The notebook still shows checkpoint/training status before samples exist.


In [ ]:
def load_manifest() -> list[dict[str, Any]]:
    if not MANIFEST_PATH.exists():
        raise FileNotFoundError(
            f'Missing {MANIFEST_PATH}. Run: python scripts/prepare_nf_sweep_aug_configs.py --project-dir "$PWD"'
        )
    return json.loads(MANIFEST_PATH.read_text())


def variant_sort_key(tag: str) -> int:
    return VARIANT_ORDER.index(tag) if tag in VARIANT_ORDER else 99


def latest_checkpoint_epoch_for_row(row: dict[str, Any]) -> int | None:
    root = Path(row.get('checkpoint_dir') or (CHECKPOINT_ROOT / f"{row['run_name']}_checkpoints"))
    epochs = []
    for path in root.glob('checkpoint-epoch-*'):
        m = re.search(r'checkpoint-epoch-(\d+)$', path.name)
        if m:
            epochs.append(int(m.group(1)))
    return max(epochs) if epochs else None


def sample_path_for_row(row: dict[str, Any], ema_label: str, sampler: str) -> Path:
    sample_root = row.get('sample_root', None)
    if sample_root is None or (isinstance(sample_root, float) and np.isnan(sample_root)):
        sample_root = SAMPLE_ROOT
    root = Path(sample_root)
    return root / f"{row['run_name']}_seed{SEED}_{ema_label}_{sampler}.npz"


def sample_path(run_name: str, ema_label: str, sampler: str) -> Path:
    # Backward-compatible helper for augmentation-sweep rows.
    return SAMPLE_ROOT / f'{run_name}_seed{SEED}_{ema_label}_{sampler}.npz'


def load_sample_array(path: Path) -> np.ndarray:
    z = np.load(path, mmap_mode='r')
    try:
        if 'samples' in z:
            return np.asarray(z['samples'])
        if 'arr_0' in z:
            return np.asarray(z['arr_0'])
        return np.asarray(z[z.files[0]])
    finally:
        z.close()


manifest = load_manifest()
if INCLUDE_NICK_DEFAULT_BASELINE:
    manifest = [NICK_DEFAULT_BASELINE, *manifest]
run_df = pd.DataFrame(manifest).sort_values('variant_tag').reset_index(drop=True)
run_df['variant_order'] = run_df['variant_tag'].map(variant_sort_key)
run_df = run_df.sort_values('variant_order').reset_index(drop=True)
run_df['checkpoint_epoch'] = run_df.apply(lambda r: latest_checkpoint_epoch_for_row(r.to_dict()), axis=1)
run_df['final_checkpoint'] = run_df['checkpoint_epoch'].eq(run_df['epochs'] - 1)
display(run_df[['run_name', 'variant_tag', 'variant_label', 'checkpoint_epoch', 'final_checkpoint', 'checkpoint_dir', 'note']])

sample_rows = []
for row in run_df.to_dict('records'):
    for ema in EMA_LABELS:
        for sampler in SAMPLER_LABELS:
            path = sample_path_for_row(row, ema, sampler)
            n_available = 0
            if path.exists():
                try:
                    n_available = len(as_nchw(load_sample_array(path)))
                except Exception as exc:
                    print('failed to inspect', path, exc)
            sample_rows.append({
                'run_name': row['run_name'],
                'variant': row['variant_tag'],
                'ema_label': ema,
                'sampler': sampler,
                'n_available': n_available,
                'exists': path.exists(),
                'sample_path': str(path),
            })

sample_df = pd.DataFrame(sample_rows)
print('sample audit:', sample_df['exists'].value_counts().to_dict())
display(sample_df.sort_values(['variant', 'ema_label', 'sampler']).reset_index(drop=True))


## Load Preferred Comparison Samples

Default comparison is `raw + train_full`, because it isolates augmentation effects without sampler/EMA changes. Set `NF_SWEEP_AUG_EMA_LABEL` or `NF_SWEEP_AUG_SAMPLER` before launching the notebook to compare another sample set.


When `NF_SWEEP_AUG_INCLUDE_NICK_DEFAULT=1` (default), the notebook also tries to load `nf_sweep_v2_u128_n500_e100_nick_default` from `results/nf_sweep_v2/samples/` with the same EMA/sampler labels.


In [ ]:
def evenly_limit(arr: np.ndarray, limit: int | None) -> np.ndarray:
    arr = np.asarray(arr)
    if limit is None or len(arr) <= limit:
        return arr.copy()
    idx = np.linspace(0, len(arr) - 1, limit, dtype=int)
    return arr[idx].copy()


real_cache: dict[str, np.ndarray] = {}
loaded: dict[str, dict[str, Any]] = {}
load_rows = []

for row in run_df.to_dict('records'):
    run_name = row['run_name']
    spath = sample_path_for_row(row, PREFERRED_EMA_LABEL, PREFERRED_SAMPLER)
    if not spath.exists():
        load_rows.append({**row, 'loaded': False, 'reason': 'missing preferred sample', 'sample_path': str(spath)})
        continue
    config_path = PROJECT_DIR / row['config']
    if not config_path.exists():
        load_rows.append({**row, 'loaded': False, 'reason': 'missing config', 'sample_path': str(spath)})
        continue
    if str(config_path) not in real_cache:
        real_cache[str(config_path)] = load_real_from_config(config_path, max_raw_samples=MAX_RAW_REAL_CUBES)
    real = as_nchw(real_cache[str(config_path)])
    generated = evenly_limit(as_nchw(load_sample_array(spath)), MAX_GENERATED)
    loaded[run_name] = {'spec': row, 'real': real, 'generated': generated, 'sample_path': spath}
    load_rows.append({
        **row,
        'loaded': True,
        'reason': '',
        'sample_path': str(spath),
        'real_shape': real.shape,
        'generated_shape': generated.shape,
    })


if loaded:
    common_n = min(MAX_GENERATED, min(len(bundle['generated']) for bundle in loaded.values()))
    if any(len(bundle['generated']) != common_n for bundle in loaded.values()):
        print(f'Using common generated count n={common_n} for fair comparison across baseline/augmentation samples.')
        for bundle in loaded.values():
            bundle['generated'] = evenly_limit(bundle['generated'], common_n)

load_df = pd.DataFrame(load_rows)
display(load_df[['run_name', 'variant_tag', 'loaded', 'reason', 'real_shape', 'generated_shape', 'sample_path'] if 'real_shape' in load_df.columns else ['run_name', 'variant_tag', 'loaded', 'reason', 'sample_path']])
print('loaded runs:', len(loaded))
if not loaded:
    print('No preferred samples loaded yet. For augmentation smoke samples run task ids 0,18,36. For Nick default, make sure the nf_sweep_v2 raw train_full sample exists.')


## One-Point Statistics

Lower `hist_l1` is better. `std_ratio` near 1 means generated field variance matches real variance.


In [ ]:
def onepoint_metrics(real: np.ndarray, generated: np.ndarray, bins: int = 120) -> dict[str, float]:
    real = evenly_limit(real, MAX_REAL_HIST)
    generated = evenly_limit(generated, MAX_GENERATED)
    rh = field_histogram(real, bins=bins)
    gh = field_histogram(generated, bins=bins)
    edges = np.asarray(rh['bin_edges'])
    width = float(np.mean(np.diff(edges)))
    hist_l1 = float(np.sum(np.abs(np.asarray(rh['hist']) - np.asarray(gh['hist']))) * width)
    return {
        'real_mean': rh['mean'],
        'generated_mean': gh['mean'],
        'real_std': rh['std'],
        'generated_std': gh['std'],
        'std_ratio': gh['std'] / max(rh['std'], 1e-30),
        'real_q01': rh['q01'],
        'generated_q01': gh['q01'],
        'real_q99': rh['q99'],
        'generated_q99': gh['q99'],
        'hist_l1': hist_l1,
    }

onepoint_rows = []
for run_name, bundle in loaded.items():
    spec = bundle['spec']
    onepoint_rows.append({
        'run_name': run_name,
        'variant': spec['variant_tag'],
        'label': VARIANT_LABELS.get(spec['variant_tag'], spec['variant_tag']),
        'n_generated': len(bundle['generated']),
        **onepoint_metrics(bundle['real'], bundle['generated']),
    })

onepoint_df = pd.DataFrame(onepoint_rows)
if len(onepoint_df):
    onepoint_df = onepoint_df.sort_values('hist_l1').reset_index(drop=True)
    display(onepoint_df)
    out = OUTPUT_DIR / 'nf_sweep_aug_onepoint_metrics.csv'
    onepoint_df.to_csv(out, index=False)
    print('wrote', out)
else:
    print('No one-point metrics yet.')


## P(k) Metrics

Lower `pk_log10_mae` is better. Ratios close to 1 are better in low/mid/high k bands.


In [ ]:
def pk_metrics(real: np.ndarray, generated: np.ndarray) -> dict[str, float]:
    real = evenly_limit(real, MAX_REAL_PK)
    generated = evenly_limit(generated, MAX_GENERATED)
    return power_spectrum_summary(real, generated, nbins=PK_NBINS)

pk_rows = []
for run_name, bundle in loaded.items():
    spec = bundle['spec']
    pk_rows.append({
        'run_name': run_name,
        'variant': spec['variant_tag'],
        'label': VARIANT_LABELS.get(spec['variant_tag'], spec['variant_tag']),
        'n_generated': len(bundle['generated']),
        **pk_metrics(bundle['real'], bundle['generated']),
    })

pk_df = pd.DataFrame(pk_rows)
if len(pk_df):
    pk_df = pk_df.sort_values('pk_log10_mae').reset_index(drop=True)
    display(pk_df)
    out = OUTPUT_DIR / 'nf_sweep_aug_pk_metrics.csv'
    pk_df.to_csv(out, index=False)
    print('wrote', out)
else:
    print('No P(k) metrics yet.')


## Physics Diagnostic Figure: One-Point And P(k) Mismatch

The top row compares one-point PDFs. The bottom row shows percentage P(k) error:

```text
100 * (P_gen(k) - P_real(k)) / P_real(k)
```


In [ ]:
if loaded:
    items = sorted(loaded.items(), key=lambda kv: variant_sort_key(kv[1]['spec']['variant_tag']))
    n = len(items)
    fig, axes = plt.subplots(2, n, figsize=(5.2 * n, 7.6), squeeze=False)
    for col, (run_name, bundle) in enumerate(items):
        label = VARIANT_LABELS.get(bundle['spec']['variant_tag'], bundle['spec']['variant_tag'])
        real = evenly_limit(bundle['real'], MAX_REAL_HIST)
        gen = evenly_limit(bundle['generated'], MAX_GENERATED)
        real_hist = field_histogram(real)
        gen_hist = field_histogram(gen)
        edges = np.asarray(real_hist['bin_edges'])
        centers = 0.5 * (edges[:-1] + edges[1:])
        ax = axes[0, col]
        ax.plot(centers, real_hist['hist'], color='black', lw=2.0, label='real')
        ax.plot(centers, gen_hist['hist'], color='tab:blue', lw=1.7, label='generated')
        ax.set_yscale('log')
        ax.set_title(f'{label}\n{PREFERRED_EMA_LABEL}, {PREFERRED_SAMPLER}')
        ax.set_xlabel('normalized field value')
        ax.set_ylabel('density')
        ax.grid(alpha=0.2)
        ax.legend()

        real_pk, kbins = batch_power_spectra(evenly_limit(bundle['real'], MAX_REAL_PK), nbins=PK_NBINS)
        gen_pk, _ = batch_power_spectra(gen, nbins=PK_NBINS)
        real_mean = np.nanmean(real_pk, axis=0)
        gen_mean = np.nanmean(gen_pk, axis=0)
        pct = 100.0 * (gen_mean - real_mean) / np.clip(real_mean, 1e-30, None)
        ax = axes[1, col]
        ax.plot(kbins, pct, marker='o', ms=3.0, color='tab:blue')
        ax.axhline(0.0, color='black', ls=':', lw=1.5)
        ax.set_xlabel('k bin')
        ax.set_ylabel('P(k) error [%]')
        ax.grid(alpha=0.2)
    fig.suptitle('u128 augmentation sweep + Nick baseline: one-point and P(k) mismatch', y=1.02)
    fig.tight_layout()
    out = OUTPUT_DIR / 'nf_sweep_aug_physics_diagnostics.png'
    fig.savefig(out, bbox_inches='tight')
    print('wrote', out)
    plt.show()
else:
    print('No loaded samples for physics diagnostic figure.')


## P(k) Curves

This view shows absolute P(k), with the real mean and a 16-84% real band.


In [ ]:
if loaded:
    items = sorted(loaded.items(), key=lambda kv: variant_sort_key(kv[1]['spec']['variant_tag']))
    n = len(items)
    fig, axes = plt.subplots(1, n, figsize=(5.2 * n, 4.2), squeeze=False)
    for ax, (run_name, bundle) in zip(axes.ravel(), items):
        label = VARIANT_LABELS.get(bundle['spec']['variant_tag'], bundle['spec']['variant_tag'])
        real_pk, kbins = batch_power_spectra(evenly_limit(bundle['real'], MAX_REAL_PK), nbins=PK_NBINS)
        gen_pk, _ = batch_power_spectra(evenly_limit(bundle['generated'], MAX_GENERATED), nbins=PK_NBINS)
        real_mean = np.nanmean(real_pk, axis=0)
        gen_mean = np.nanmean(gen_pk, axis=0)
        lo, hi = np.nanpercentile(real_pk, [16, 84], axis=0)
        ax.fill_between(kbins, lo, hi, color='black', alpha=0.14, label='real 16-84%')
        ax.plot(kbins, real_mean, color='black', lw=2.0, label='real mean')
        ax.plot(kbins, gen_mean, color='tab:blue', marker='o', ms=3.0, lw=1.7, label='generated')
        ax.set_yscale('log')
        ax.set_title(label)
        ax.set_xlabel('k bin')
        ax.set_ylabel('P(k)')
        ax.grid(alpha=0.2)
        ax.legend()
    fig.suptitle('u128 augmentation sweep + Nick baseline: P(k)', y=1.02)
    fig.tight_layout()
    out = OUTPUT_DIR / 'nf_sweep_aug_pk_curves.png'
    fig.savefig(out, bbox_inches='tight')
    print('wrote', out)
    plt.show()
else:
    print('No loaded samples for P(k) curves.')


## Real vs Generated Images

Each column uses one real slice and one generated sample from the same variant. The color scale is shared across the two images in that column.


In [ ]:
def image_limits(*arrays: np.ndarray) -> tuple[float, float]:
    vals = np.concatenate([np.asarray(a).ravel() for a in arrays])
    lo, hi = np.nanpercentile(vals, [1, 99])
    if not np.isfinite(lo) or not np.isfinite(hi) or lo == hi:
        return -1.0, 1.0
    return float(lo), float(hi)

if loaded:
    items = sorted(loaded.items(), key=lambda kv: variant_sort_key(kv[1]['spec']['variant_tag']))
    n = len(items)
    fig, axes = plt.subplots(2, n, figsize=(4.2 * n, 7.2), squeeze=False)
    for col, (run_name, bundle) in enumerate(items):
        label = VARIANT_LABELS.get(bundle['spec']['variant_tag'], bundle['spec']['variant_tag'])
        real_img = bundle['real'][0, 0]
        gen_img = bundle['generated'][0, 0]
        vmin, vmax = image_limits(real_img, gen_img)
        axes[0, col].imshow(real_img, origin='lower', cmap='viridis', vmin=vmin, vmax=vmax)
        axes[0, col].set_title(f'{label}\nreal')
        axes[1, col].imshow(gen_img, origin='lower', cmap='viridis', vmin=vmin, vmax=vmax)
        axes[1, col].set_title('generated')
        for ax in (axes[0, col], axes[1, col]):
            ax.set_xticks([])
            ax.set_yticks([])
    fig.suptitle('u128 augmentation sweep + Nick baseline: real vs generated images', y=1.02)
    fig.tight_layout()
    out = OUTPUT_DIR / 'nf_sweep_aug_images.png'
    fig.savefig(out, bbox_inches='tight')
    print('wrote', out)
    plt.show()
else:
    print('No loaded samples for image grid.')


## Training Loss Curves

Training loss is only an optimization-health check. It does not decide sample quality; compare it with one-point and P(k).


In [ ]:
def metric_candidates(run_name: str) -> list[Path]:
    root = CHECKPOINT_ROOT / f'{run_name}_checkpoints'
    paths = []
    paths.extend(sorted(root.glob('metrics_epoch_*.json')))
    paths.extend(sorted(root.glob('metrics.json')))
    for ckpt in sorted(root.glob('checkpoint-epoch-*')):
        paths.extend(sorted(ckpt.glob('metrics*.json')))
    return paths


def moving_average(values: np.ndarray, window: int) -> np.ndarray:
    values = np.asarray(values, dtype=float)
    if len(values) == 0 or window <= 1:
        return values
    window = min(window, len(values))
    kernel = np.ones(window, dtype=float) / window
    return np.convolve(values, kernel, mode='valid')

metrics_by_run = {}
metric_rows = []
for row in run_df.to_dict('records'):
    paths = metric_candidates(row['run_name'])
    metrics = None
    if paths:
        with paths[-1].open() as f:
            metrics = json.load(f)
        metrics_by_run[row['run_name']] = metrics
    epoch_loss = np.asarray((metrics or {}).get('epoch_loss', []), dtype=float)
    metric_rows.append({
        'run_name': row['run_name'],
        'variant': row['variant_tag'],
        'metrics_path': str(paths[-1]) if paths else None,
        'n_epochs_logged': len(epoch_loss),
        'latest_epoch_loss': float(epoch_loss[-1]) if len(epoch_loss) else np.nan,
        'best_epoch_loss': float(np.nanmin(epoch_loss)) if len(epoch_loss) else np.nan,
    })
metrics_df = pd.DataFrame(metric_rows)
display(metrics_df)

if metrics_by_run:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
    for row in run_df.sort_values('variant_order').to_dict('records'):
        metrics = metrics_by_run.get(row['run_name'])
        if not metrics:
            continue
        label = VARIANT_LABELS.get(row['variant_tag'], row['variant_tag'])
        batch_loss = np.asarray(metrics.get('loss', []), dtype=float)
        epoch_loss = np.asarray(metrics.get('epoch_loss', []), dtype=float)
        epoch_lr = np.asarray(metrics.get('epoch_lr', []), dtype=float)
        if len(batch_loss):
            window = max(1, len(batch_loss) // 500)
            y = moving_average(batch_loss, window)
            axes[0].plot(np.arange(len(y)), y, label=label)
        if len(epoch_loss):
            axes[1].plot(np.arange(len(epoch_loss)), epoch_loss, marker='o', ms=2.5, label=label)
        if len(epoch_lr):
            axes[2].plot(np.arange(len(epoch_lr)), epoch_lr, marker='o', ms=2.5, label=label)
    axes[0].set_title('batch loss')
    axes[0].set_xlabel('optimizer step')
    axes[0].set_ylabel('MSE loss')
    axes[1].set_title('epoch loss')
    axes[1].set_xlabel('epoch')
    axes[1].set_ylabel('mean MSE loss')
    axes[2].set_title('learning rate')
    axes[2].set_xlabel('epoch')
    axes[2].set_ylabel('LR')
    for ax in axes:
        ax.grid(alpha=0.25)
        ax.set_yscale('log')
    fig.legend(loc='lower center', bbox_to_anchor=(0.5, -0.08), ncol=3)
    fig.suptitle('u128 augmentation sweep: training curves', y=1.02)
    fig.tight_layout(rect=(0, 0.10, 1, 0.98))
    out = OUTPUT_DIR / 'nf_sweep_aug_training_curves.png'
    fig.savefig(out, bbox_inches='tight')
    print('wrote', out)
    plt.show()
else:
    print('No training metrics JSON found yet.')


## Optional EMA/Sampler Ranking

If the full sampling array has run, this cell scores every available sample file and reports the best EMA/sampler per augmentation variant by P(k) and one-point error.


In [ ]:
rank_rows = []
for row in run_df.to_dict('records'):
    config_path = PROJECT_DIR / row['config']
    if str(config_path) not in real_cache and config_path.exists():
        real_cache[str(config_path)] = load_real_from_config(config_path, max_raw_samples=MAX_RAW_REAL_CUBES)
    real = as_nchw(real_cache[str(config_path)]) if str(config_path) in real_cache else None
    if real is None:
        continue
    for ema in EMA_LABELS:
        for sampler in SAMPLER_LABELS:
            path = sample_path_for_row(row, ema, sampler)
            if not path.exists():
                continue
            generated = evenly_limit(as_nchw(load_sample_array(path)), MAX_GENERATED)
            op = onepoint_metrics(real, generated)
            pk = pk_metrics(real, generated)
            rank_rows.append({
                'run_name': row['run_name'],
                'variant': row['variant_tag'],
                'label': VARIANT_LABELS.get(row['variant_tag'], row['variant_tag']),
                'ema_label': ema,
                'sampler': sampler,
                'n_generated': len(generated),
                'hist_l1': op['hist_l1'],
                'std_ratio': op['std_ratio'],
                **pk,
                'sample_path': str(path),
            })

rank_df = pd.DataFrame(rank_rows)
if len(rank_df):
    display(rank_df.sort_values(['variant', 'pk_log10_mae']).reset_index(drop=True))
    best_pk = rank_df.sort_values(['variant', 'pk_log10_mae']).groupby('variant', as_index=False).head(1)
    best_hist = rank_df.sort_values(['variant', 'hist_l1']).groupby('variant', as_index=False).head(1)
    print('Best by P(k):')
    display(best_pk[['variant', 'ema_label', 'sampler', 'n_generated', 'pk_log10_mae', 'hist_l1', 'std_ratio']])
    print('Best by one-point histogram:')
    display(best_hist[['variant', 'ema_label', 'sampler', 'n_generated', 'hist_l1', 'pk_log10_mae', 'std_ratio']])
    out = OUTPUT_DIR / 'nf_sweep_aug_all_sample_metrics.csv'
    rank_df.to_csv(out, index=False)
    print('wrote', out)
else:
    print('No available sample files for EMA/sampler ranking yet.')


## Practical Interpretation

Use this sweep to answer one specific question: does exact square-symmetry augmentation improve the u128 Nick-default-like recipe? Compare against `nick_default` as the older nf_sweep_v2 reference, and compare the three augmentation variants against each other as the cleaner controlled ablation.

- If `roll_flip_rot90` or `roll_dihedral` improves both `hist_l1` and `pk_log10_mae`, the new augmentation should be worth a PR.
- If P(k) improves but one-point gets worse, it is a tradeoff and should be sampled more before judging.
- If training loss improves but samples do not, do not trust the loss by itself.
- `roll_dihedral` is the cleanest PR story: one exact D4 operation, no interpolation, physically motivated for isotropic square CAMELS maps.
